In [6]:
# Training the Probability of Default (PD) Model
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

# 1. Load the booked portfolio
data_path = os.path.join('..', 'data', 'processed', 'booked_portfolio.csv')
df = pd.read_csv(data_path)

# 2. Select Features (X) and Target (y)
# We use standard risk drivers available at origination
features = ['cibil_score', 'dti_ratio', 'monthly_income_inr', 'loan_amount_inr']
X = df[features]
y = df['default_flag']

# 3. Train-Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train Logistic Regression Model
# Using 'balanced' class_weight because defaults are typically a minority class
pd_model = LogisticRegression(class_weight='balanced', random_state=42)
pd_model.fit(X_train_scaled, y_train)

# 5. Evaluate the Model
y_pred_proba = pd_model.predict_proba(X_test_scaled)[:, 1]
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f"PD Model AUC-ROC Score: {auc_score:.3f}")
print("\nClassification Report (Test Data):")
print(classification_report(y_test, pd_model.predict(X_test_scaled)))

# 6. Apply PD to the entire portfolio
# We score the entire dataset to calculate portfolio-level ECL
X_all_scaled = scaler.transform(X)
df['PD'] = pd_model.predict_proba(X_all_scaled)[:, 1]

PD Model AUC-ROC Score: 0.566

Classification Report (Test Data):
              precision    recall  f1-score   support

           0       0.97      0.54      0.69      7566
           1       0.04      0.54      0.07       243

    accuracy                           0.54      7809
   macro avg       0.50      0.54      0.38      7809
weighted avg       0.94      0.54      0.67      7809



In [7]:
# Calculating Expected Credit Loss (ECL)
# Loss Given Default (LGD) varies heavily by the collateral backing the loan. Unsecured loans recover less than secured ones.

# 1. Assign Loss Given Default (LGD) based on Indian retail standards
def assign_lgd(product):
    if product == 'Personal Loan': 
        return 0.65  # 65% loss on unsecured PLs
    elif product == 'Two-Wheeler': 
        return 0.45  # 45% loss (vehicle can be repossessed)
    else: 
        return 0.55  # Consumer Durables

df['LGD'] = df['loan_product'].apply(assign_lgd)

# 2. Assign Exposure at Default (EAD)
# For this static portfolio snapshot, we assume EAD is the current loan amount.
df['EAD'] = df['loan_amount_inr']

# 3. Calculate IND-AS 109 Expected Credit Loss
df['ECL_Amount'] = df['PD'] * df['LGD'] * df['EAD']

# 4. Portfolio Level Summary for the CRO Dashboard
total_exposure = df['EAD'].sum()
total_ecl_provision = df['ECL_Amount'].sum()
portfolio_ecl_pct = (total_ecl_provision / total_exposure) * 100

print("\n--- Portfolio ECL Summary ---")
print(f"Total Portfolio Exposure (EAD): ₹{total_exposure:,.2f}")
print(f"Total Expected Credit Loss (ECL): ₹{total_ecl_provision:,.2f}")
print(f"Overall Provisioning Rate: {portfolio_ecl_pct:.2f}%")

# Save final enriched dataset for the dashboard
df.to_csv(os.path.join('..', 'data', 'processed', 'portfolio_with_ecl.csv'), index=False)


--- Portfolio ECL Summary ---
Total Portfolio Exposure (EAD): ₹3,781,579,000.00
Total Expected Credit Loss (ECL): ₹1,085,992,094.76
Overall Provisioning Rate: 28.72%
